In [5]:
import simnibs
from simnibs import mesh_io
import numpy as np
import nibabel as nib
import pandas as pd
import os
#from scipy.spatial.distance import cdist
from simnibs.simulation import electrode_placement
#from simnibs.cli import msh2nii
import subprocess

In [6]:
def create_3x3_array(center, normal):
    """Create a 3x3 grid of electrodes on a plane"""
    if abs(normal[0]) > 0.5:
        temp = np.array([0, 1, 0])
    else:
        temp = np.array([1, 0, 0])
    
    tangent1 = np.cross(normal, temp)
    tangent1 /= np.linalg.norm(tangent1)
    tangent2 = np.cross(normal, tangent1)
    
    positions = []
    for i in [-1, 0, 1]:
        for j in [-1, 0, 1]:
            offset = (i * 25 * tangent1) + (j * 27 * tangent2)
            positions.append(center + offset)
    
    return np.array(positions)

In [87]:
def get_random_scalp_locations(mesh, min_distance=100, max_attempts=100):
    """Get two random scalp locations that aren't too close"""
    scalp_surface = mesh.crop_mesh(1005)
    nodes = scalp_surface.nodes.node_coord
    normals = scalp_surface.nodes_normals()

   # Get min/max for Z (height) and Y (depth)
    z_min, z_max = np.min(nodes[:, 2]), np.max(nodes[:, 2])  # Z: inferior-superior
    y_min, y_max = np.min(nodes[:, 1]), np.max(nodes[:, 1])  # Y: posterior-anterior
    
    # Define thresholds as percentages (e.g., 30% from bottom, 20% from front)
    z_threshold = z_min + 0.55 * (z_max - z_min)  # Keep nodes above 45% of head height
    y_threshold = y_min + 0.20 * (y_max - y_min)  # Keep nodes behind 20% of head depth
    z2_threshold = z_min + 35

    
   
    # Combined condition: above eyebrows OR behind ears and with normal not directed into -z
    mask = ((nodes[:, 2] > z_threshold) | ((nodes[:, 1] < y_threshold) & (nodes[:, 2] > z2_threshold)))
    # Filter nodes and normals
    filtered_nodes = nodes[mask]
    filtered_normals = normals[mask]

    x_min, x_max = np.min(filtered_nodes[:, 0]), np.max(filtered_nodes[:, 0])
    # Get indices of nodes with x_min
    idx1 = np.argmin(filtered_nodes[:, 0])
    idx2 = np.argmax(filtered_nodes[:, 0])
    # Pick first random location
    #idx1 = np.random.choice(len(filtered_nodes))
    center1 = filtered_nodes[idx1]
    normal1 = -filtered_normals[idx1]  # Flip normal inward
    normal1 /= np.linalg.norm(normal1)
    
    # Pick second location that's sufficiently far away
    #attempts = 0
    #while attempts < max_attempts:
        #idx2 = np.random.choice(len(filtered_nodes))
    center2 = filtered_nodes[idx2]
        
        # Check distance between array centers
        #if np.linalg.norm(center1 - center2) >= min_distance:
    normal2 = -filtered_normals[idx2]
    normal2 /= np.linalg.norm(normal2)
    return [(center1, normal1), (center2, normal2)]
        
    
  

In [8]:
def add_electrode_arrays(mesh, array_locations):
    """Add electrode arrays to the mesh"""
    #all_electrodes_poses = []
    for array_idx, (center, normal) in enumerate(array_locations):
        positions = create_3x3_array(center, normal)
        
        for i, pos in enumerate(positions):
            el = simnibs.simulation.sim_struct.ELECTRODE()
            el.centre = pos
            el.shape = 'ellipse'
            el.dimensions = [20, 20]  # 20mm diameter
            el.thickness = [2 , 2]  # 2mm gel 2mm elecrode
            #all_electrodes_poses.append(pos)
            mesh_with_elec , tag = electrode_placement.put_electrode_on_mesh(
                el, mesh, 
                array_idx + 2000) #tag
    
    return mesh_with_elec 

In [88]:
input_mesh_path = "D:/Results/FEM/mesh/test_mesh27_2.msh"
final_path = "D:/Results/FEM/mesh with electrodes/test_mesh27_2.msh"


ref_nii_path = "D:/Results/FEM/nifti labeled/labeled_FEM-Test_TCGA-02-0027.nii.gz"
ref_img = nib.load(ref_nii_path)

mesh = mesh_io.read_msh(input_mesh_path)
array_locations = get_random_scalp_locations(mesh)
mesh_with_elec = add_electrode_arrays(mesh, array_locations)

# Save final model
mesh_io.write_msh(mesh_with_elec, final_path)

output_label = "D:/Results/FEM/mesh with electrodes/test_labels27_2.nii.gz"

command = f'msh2nii "{final_path}" "{ref_nii_path}" "{output_label}" --create_label'
subprocess.run(command)
print("done")

done


In [89]:
mesh_path = "D:/Results/FEM/mesh with electrodes/test_mesh27_2.msh"
result_mesh = mesh_io.read_msh(mesh_path)
#Electrode = result_mesh.crop_mesh(3400)
v = result_mesh.view()
v.show()

In [19]:
help(mesh_io.Elements)

Help on class Elements in module simnibs.mesh_tools.mesh_io:

class Elements(builtins.object)
 |  Elements(triangles=None, tetrahedra=None)
 |  
 |  Mesh elements.
 |  
 |  Can only handle triangles and tetrahedra!
 |  
 |  Parameters
 |  --------------------------
 |  triangles (optional): (Nx3) ndarray
 |      List of nodes composing each triangle
 |  tetrahedra(optional): (Nx3) ndarray
 |      List of nodes composing each tetrahedra
 |  
 |  
 |  Attributes
 |  ----------------------------------
 |  elm_number: (Nx1) ndarray
 |        element ID (u from 1 till nr)
 |  elm_type: (Nx1) ndarray
 |      elm-type (2=triangle, 4=tetrahedron, etc)
 |  tag1: (Nx1) ndarray
 |      first tag for each element
 |  tag2: (Nx1) ndarray
 |      second tag for each elenent
 |  node_number_list: (Nx4) ndarray
 |      4xnumber_of_element matrix of the nodes that constitute the element.
 |      For the triangles, the fourth element = -1
 |  nr: int
 |      Number or elemets
 |  
 |  
 |  Notes
 |  ---